# Day 11 — Mock Interview Practice

**Week 2 · Day 4 · Oct 23, 2026** — Difficulty: **INTERMEDIATE → ADVANCED** (7 INTERMEDIATE, 11 ADVANCED)

18 questions across 8 categories. Attempt every question on paper or a scratch cell **before** opening the solution. Time yourself: INTERMEDIATE ≈ 15 min, ADVANCED ≈ 25 min.

| # | Section | Q | Topics |
|---|---|---|---|
| 1 | SQL Advanced | Q1–Q2 | session detection via gaps-and-islands · cohort retention matrix |
| 2 | Python — Loops/Function/List/Dict | Q3–Q4 | LRU cache (OrderedDict + doubly-linked list) · sliding window — longest substring with K distinct chars |
| 3 | PySpark Coding | Q5–Q6 | window functions for MoM growth & ranking · CDC processing with MERGE INTO for SCD Type 2 |
| 4 | Spark Optimization | Q7–Q8 | broadcast vs sort-merge join threshold tuning · AQE internals — coalescing, skew, dynamic partition pruning |
| 5 | ETL — Failures, Late Data, Schema | Q9–Q11 | exactly-once with checkpointing · dead letter queue for poison messages · Delta OPTIMIZE & ZORDER |
| 6 | Data Modeling | Q12–Q13 | SCD Type 1/2 with MERGE logic · late-arriving facts vs dimensions |
| 7 | Snowflake | Q14–Q15 | time travel & zero-copy cloning · materialized vs dynamic tables & search optimization |
| 8 | Monitoring, Logging & Orchestration | Q16–Q18 | Jobs orchestration (task deps, Run If, triggers) · pipeline vs data-level monitoring · 3 AM failure debugging walkthrough |

**Score card at the bottom.** Fill it honestly — it drives what you review tomorrow.

## Q1 — Session Detection via Gaps-and-Islands `[INTERMEDIATE]`

You are given a `clickstream` table with `user_id`, `event_ts` (timestamp), `event_type`. Product analytics wants to identify **browsing sessions**: a new session starts when there is a gap of **more than 30 minutes** since the user's previous event.

**Output per session:** `user_id`, `session_id` (sequential per user starting at 1), `session_start`, `session_end`, `num_events`, `session_duration_min`.

**Requirements:**
* Use the classic gaps-and-islands technique: `LAG` to compute the time gap, a conditional flag, and a cumulative `SUM` to create island IDs.
* Sort by `user_id, event_ts`.

**Sample data:**

| user_id | event_ts | event_type |
|---------|----------|------------|
| u1 | 2026-10-23 09:00:00 | view |
| u1 | 2026-10-23 09:10:00 | click |
| u1 | 2026-10-23 09:45:00 | view |
| u1 | 2026-10-23 09:50:00 | click |
| u2 | 2026-10-23 10:00:00 | view |
| u2 | 2026-10-23 10:05:00 | click |

**Expected:** u1: session 1 (09:00-09:10, 2 events, 10 min), session 2 (09:45-09:50, 2 events, 5 min); u2: session 1 (10:00-10:05, 2 events, 5 min)

In [0]:
# Solution Q1 — Gaps-and-Islands: session detection with LAG + cumulative SUM
from pyspark.sql import functions as F, types as T, Window as W

schema = T.StructType([
    T.StructField("user_id", T.StringType()),
    T.StructField("event_ts", T.StringType()),
    T.StructField("event_type", T.StringType()),
])
clickstream = spark.createDataFrame([
    ("u1", "2026-10-23 09:00:00", "view"),
    ("u1", "2026-10-23 09:10:00", "click"),
    ("u1", "2026-10-23 09:45:00", "view"),
    ("u1", "2026-10-23 09:50:00", "click"),
    ("u2", "2026-10-23 10:00:00", "view"),
    ("u2", "2026-10-23 10:05:00", "click"),
], schema)
clickstream = clickstream.withColumn("event_ts", F.to_timestamp("event_ts"))
clickstream.createOrReplaceTempView("clickstream")

result = spark.sql("""
WITH gaps AS (
  SELECT user_id, event_ts, event_type,
    BIGINT(unix_timestamp(event_ts) - unix_timestamp(
      LAG(event_ts) OVER (PARTITION BY user_id ORDER BY event_ts)
    )) AS gap_secs
  FROM clickstream
),
flagged AS (
  SELECT user_id, event_ts, event_type,
    CASE WHEN gap_secs IS NULL OR gap_secs > 1800 THEN 1 ELSE 0 END AS is_new_session
  FROM gaps
),
islands AS (
  SELECT user_id, event_ts, event_type,
    SUM(is_new_session) OVER (PARTITION BY user_id ORDER BY event_ts
         ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS session_id
  FROM flagged
)
SELECT user_id, session_id,
  MIN(event_ts) AS session_start, MAX(event_ts) AS session_end,
  COUNT(*) AS num_events,
  ROUND(BIGINT(unix_timestamp(MAX(event_ts)) - unix_timestamp(MIN(event_ts))) / 60.0, 1) AS session_duration_min
FROM islands
GROUP BY user_id, session_id
ORDER BY user_id, session_id
""")
display(result)
rows = result.collect()
assert len(rows) == 3, f"Expected 3 sessions, got {len(rows)}"
u1_s1 = [r for r in rows if r['user_id'] == 'u1' and r['session_id'] == 1][0]
assert u1_s1['num_events'] == 2 and u1_s1['session_duration_min'] == 10.0
u2_s1 = [r for r in rows if r['user_id'] == 'u2' and r['session_id'] == 1][0]
assert u2_s1['num_events'] == 2 and u2_s1['session_duration_min'] == 5.0
print("Q1 all assertions passed")

## Q2 — Cohort Retention Matrix `[ADVANCED]`

You have a `user_events` table with `user_id`, `event_date`, `event_type` (`signup`, `active`). The **acquisition cohort** is the month of a user's first `signup`. Retention in month N = fraction of cohort members who had at least one `active` event in the Nth month after signup.

**Output:** A retention matrix — one row per cohort month, columns `cohort_month`, `cohort_size`, `m0` (signup month, always 100%), `m1`, `m2`, `m3`.

**Sample data:**

| user_id | event_date | event_type |
|---------|-----------|------------|
| u1 | 2026-08-05 | signup | u1 | 2026-08-10 | active | u1 | 2026-09-15 | active |
| u2 | 2026-08-20 | signup | u2 | 2026-08-25 | active |
| u3 | 2026-09-01 | signup | u3 | 2026-09-05 | active | u3 | 2026-10-10 | active |
| u4 | 2026-08-01 | signup | (no activity after signup)

**Expected:** Aug cohort: size 3, m0=100%, m1=33.3% (only u1 active in Sep); Sep cohort: size 1, m0=100%, m1=100%

In [0]:
# Solution Q2 — Cohort retention matrix with conditional aggregation + pivot
from pyspark.sql import functions as F, types as T

schema = T.StructType([
    T.StructField("user_id", T.StringType()),
    T.StructField("event_date", T.StringType()),
    T.StructField("event_type", T.StringType()),
])
user_events = spark.createDataFrame([
    ("u1", "2026-08-05", "signup"), ("u1", "2026-08-10", "active"), ("u1", "2026-09-15", "active"),
    ("u2", "2026-08-20", "signup"), ("u2", "2026-08-25", "active"),
    ("u3", "2026-09-01", "signup"), ("u3", "2026-09-05", "active"), ("u3", "2026-10-10", "active"),
    ("u4", "2026-08-01", "signup"),
], schema)
user_events = user_events.withColumn("event_date", F.to_date("event_date"))
user_events.createOrReplaceTempView("user_events")

result = spark.sql("""
WITH signups AS (
  SELECT user_id, DATE_TRUNC('month', event_date) AS cohort_month
  FROM user_events WHERE event_type = 'signup'
  QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY event_date) = 1
),
cohort_size AS (
  SELECT cohort_month, COUNT(DISTINCT user_id) AS size FROM signups GROUP BY cohort_month
),
retention AS (
  SELECT s.cohort_month,
    CAST(MONTHS_BETWEEN(a.event_date, s.cohort_month) AS INT) AS mo,
    COUNT(DISTINCT s.user_id) AS active_users
  FROM signups s
  JOIN user_events a ON s.user_id = a.user_id AND a.event_type = 'active'
  WHERE MONTHS_BETWEEN(a.event_date, s.cohort_month) BETWEEN 0 AND 3
  GROUP BY s.cohort_month, mo
)
SELECT c.cohort_month, c.size AS cohort_size,
  100.0 AS m0,  -- signup month is always 100% by definition
  ROUND(100.0 * COALESCE(r1.active_users, 0) / c.size, 1) AS m1,
  ROUND(100.0 * COALESCE(r2.active_users, 0) / c.size, 1) AS m2,
  ROUND(100.0 * COALESCE(r3.active_users, 0) / c.size, 1) AS m3
FROM cohort_size c
LEFT JOIN retention r0 ON c.cohort_month = r0.cohort_month AND r0.mo = 0
LEFT JOIN retention r1 ON c.cohort_month = r1.cohort_month AND r1.mo = 1
LEFT JOIN retention r2 ON c.cohort_month = r2.cohort_month AND r2.mo = 2
LEFT JOIN retention r3 ON c.cohort_month = r3.cohort_month AND r3.mo = 3
ORDER BY c.cohort_month
""")
display(result)
rows = {r['cohort_month'].strftime('%Y-%m'): r for r in result.collect()}
aug = rows['2026-08']
assert aug['cohort_size'] == 3 and float(aug['m0']) == 100.0 and float(aug['m1']) == 33.3
sep = rows['2026-09']
assert sep['cohort_size'] == 1 and float(sep['m0']) == 100.0 and float(sep['m1']) == 100.0
print("Q2 all assertions passed")

## Q3 — LRU Cache `[INTERMEDIATE]`

Implement an **LRU (Least Recently Used) cache** with two approaches:

1. **Using `collections.OrderedDict`** — `get(key)` and `put(key, value)` both O(1). When capacity is exceeded, evict the least recently used item.
2. **Using a doubly-linked list + hash map** (manual implementation) — demonstrates the internals.

**API:** `LRUCache(capacity)`, `get(key) -> int` (return -1 if not found), `put(key, value) -> None`

**Test:** capacity=2: put(1,1); put(2,2); get(1)->1; put(3,3)->evicts key 2; get(2)->-1; put(4,4)->evicts key 1; get(1)->-1; get(3)->3; get(4)->4
(LeetCode 146 — Amazon, Google, Microsoft)

In [0]:
# Solution Q3 — LRU Cache: OrderedDict version + manual doubly-linked list version
from collections import OrderedDict

class LRUCacheOrderedDict:
    def __init__(self, capacity):
        self.cache = OrderedDict()
        self.capacity = capacity
    def get(self, key):
        if key not in self.cache: return -1
        self.cache.move_to_end(key)
        return self.cache[key]
    def put(self, key, value):
        if key in self.cache: self.cache.move_to_end(key)
        self.cache[key] = value
        if len(self.cache) > self.capacity: self.cache.popitem(last=False)

class _Node:
    __slots__ = ('key', 'val', 'prev', 'next')
    def __init__(self, key=0, val=0):
        self.key, self.val, self.prev, self.next = key, val, None, None

class LRUCache:
    def __init__(self, capacity):
        self.capacity, self.map = capacity, {}
        self.head, self.tail = _Node(), _Node()
        self.head.next, self.tail.prev = self.tail, self.head
    def _remove(self, n): n.prev.next, n.next.prev = n.next, n.prev
    def _add_tail(self, n):
        n.prev, n.next = self.tail.prev, self.tail
        self.tail.prev.next, self.tail.prev = n, n
    def get(self, key):
        if key not in self.map: return -1
        n = self.map[key]; self._remove(n); self._add_tail(n); return n.val
    def put(self, key, value):
        if key in self.map: self._remove(self.map[key])
        n = _Node(key, value); self.map[key] = n; self._add_tail(n)
        if len(self.map) > self.capacity:
            lru = self.head.next; self._remove(lru); del self.map[lru.key]

def test_cache(cls):
    c = cls(2)
    c.put(1, 1); c.put(2, 2); assert c.get(1) == 1
    c.put(3, 3); assert c.get(2) == -1
    c.put(4, 4); assert c.get(1) == -1
    assert c.get(3) == 3; assert c.get(4) == 4
    c.put(3, 30); assert c.get(3) == 30

for cls in (LRUCacheOrderedDict, LRUCache): test_cache(cls)
print("Q3 LRU Cache — both implementations passed")

## Q4 — Longest Substring with At Most K Distinct Characters `[ADVANCED]`

Given a string `s` and an integer `k`, find the length of the **longest substring** that contains **at most `k` distinct characters**.

**Examples:** `s="eceba", k=2` -> 3 (`"ece"`); `s="ccaabbb", k=2` -> 5 (`"aabbb"`); `s="aabbcc", k=3` -> 6; `s="abc", k=0` -> 0

Use **sliding window / two-pointer** with a hash map. O(n) time, O(k) space. Also implement a variant returning the substring itself.
(LeetCode 340 — Google, Amazon, Meta)

In [0]:
# Solution Q4 — Sliding window: longest substring with at most K distinct characters
from collections import defaultdict

def longest_substring_k_distinct(s, k):
    if k == 0 or not s: return 0
    freq = defaultdict(int); left = max_len = 0
    for right, ch in enumerate(s):
        freq[ch] += 1
        while len(freq) > k:
            lc = s[left]; freq[lc] -= 1
            if freq[lc] == 0: del freq[lc]
            left += 1
        max_len = max(max_len, right - left + 1)
    return max_len

def longest_substring_k_distinct_str(s, k):
    if k == 0 or not s: return ""
    freq = defaultdict(int); left = best_start = best_len = 0
    for right, ch in enumerate(s):
        freq[ch] += 1
        while len(freq) > k:
            lc = s[left]; freq[lc] -= 1
            if freq[lc] == 0: del freq[lc]
            left += 1
        if right - left + 1 > best_len: best_start, best_len = left, right - left + 1
    return s[best_start:best_start + best_len]

assert longest_substring_k_distinct("eceba", 2) == 3
assert longest_substring_k_distinct("ccaabbb", 2) == 5
assert longest_substring_k_distinct("aabbcc", 3) == 6
assert longest_substring_k_distinct("abc", 0) == 0
assert longest_substring_k_distinct("", 5) == 0
assert longest_substring_k_distinct("aaaa", 1) == 4
assert longest_substring_k_distinct_str("eceba", 2) == "ece"
assert longest_substring_k_distinct_str("ccaabbb", 2) == "aabbb"
assert longest_substring_k_distinct_str("aabbcc", 3) == "aabbcc"
print("Q4 all assertions passed")

## Q5 — Window Functions for MoM Growth & Ranking `[INTERMEDIATE]`

You have a `monthly_sales` table with `category`, `month` (`YYYY-MM`), `revenue`. Product teams want:

1. **Month-over-month growth rate** per category: `(current - prev) / prev * 100`
2. **Running total** (cumulative revenue) per category across months
3. **Rank within each month** by revenue (dense rank — ties get same rank)

**Output columns:** `category, month, revenue, mom_growth_pct, running_total, month_rank`

In [0]:
# Solution Q5 — Window functions: MoM growth, running total, dense rank
from pyspark.sql import functions as F, types as T

monthly_sales = spark.createDataFrame([
    ("Electronics", "2026-08", 10000), ("Electronics", "2026-09", 12000), ("Electronics", "2026-10", 9000),
    ("Books", "2026-08", 5000), ("Books", "2026-09", 5000), ("Books", "2026-10", 7000),
], "category STRING, month STRING, revenue INT")
monthly_sales.createOrReplaceTempView("monthly_sales")

result = spark.sql("""
WITH metrics AS (
  SELECT category, month, revenue,
    CASE WHEN LAG(revenue) OVER w IS NULL THEN NULL
      ELSE ROUND((revenue - LAG(revenue) OVER w) / LAG(revenue) OVER w * 100, 2)
    END AS mom_growth_pct,
    SUM(revenue) OVER (PARTITION BY category ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
    DENSE_RANK() OVER (PARTITION BY month ORDER BY revenue DESC) AS month_rank
  FROM monthly_sales
  WINDOW w AS (PARTITION BY category ORDER BY month)
)
SELECT * FROM metrics ORDER BY category, month
""")
display(result)
rows = {(r['category'], r['month']): r for r in result.collect()}
e_sep = rows[('Electronics', '2026-09')]
assert e_sep['mom_growth_pct'] == 20.0 and e_sep['running_total'] == 22000 and e_sep['month_rank'] == 1
b_sep = rows[('Books', '2026-09')]
assert b_sep['mom_growth_pct'] == 0.0 and b_sep['running_total'] == 10000
assert rows[('Electronics', '2026-08')]['mom_growth_pct'] is None
print("Q5 all assertions passed")

## Q6 — CDC Processing with MERGE INTO for SCD Type 2 `[ADVANCED]`

You maintain a `dim_customers` Delta table with SCD Type 2: `customer_id`, `name`, `city`, `email`, `is_current`, `effective_from`, `effective_to` (null = current).

A daily CDC feed arrives. You must: (1) **Expire** existing current records whose attributes changed, (2) **Insert** new current records for changed customers, (3) **Upsert** unchanged or new customers.

Use a single `MERGE INTO`. Show the Delta table before/after. Verify: changed customers have 2 rows (old expired + new current), unchanged keep 1 row.

In [0]:
# Solution Q6 — CDC MERGE INTO for SCD Type 2 on Delta table
from pyspark.sql import functions as F
from delta.tables import DeltaTable

tbl = "workspace.default.day11_q6_dim_customers"
spark.sql(f"DROP TABLE IF EXISTS {tbl}")

spark.createDataFrame([
    (1, "Alice", "NYC", "alice@old.com", True, "2026-01-01", None),
    (2, "Bob", "LA", "bob@x.com", True, "2026-01-01", None),
    (3, "Carol", "SF", "carol@x.com", True, "2026-01-01", None),
], "customer_id INT, name STRING, city STRING, email STRING, is_current BOOLEAN, effective_from STRING, effective_to STRING").write.mode("overwrite").saveAsTable(tbl)

cdc = spark.createDataFrame([
    (1, "Alice", "Boston", "alice@new.com", "2026-10-23"),
    (2, "Bob", "LA", "bob@x.com", "2026-10-23"),
    (4, "Dave", "Chicago", "dave@x.com", "2026-10-23"),
], "customer_id INT, name STRING, city STRING, email STRING, cdc_date STRING")
cdc.createOrReplaceTempView("customer_cdc")

spark.sql(f"""
MERGE INTO {tbl} AS t
USING customer_cdc AS s ON t.customer_id = s.customer_id AND t.is_current = true
WHEN MATCHED AND (t.name != s.name OR t.city != s.city OR t.email != s.email)
  THEN UPDATE SET t.is_current = false, t.effective_to = s.cdc_date
WHEN NOT MATCHED
  THEN INSERT (customer_id, name, city, email, is_current, effective_from, effective_to)
  VALUES (s.customer_id, s.name, s.city, s.email, true, s.cdc_date, null)
""")

spark.sql(f"""
INSERT INTO {tbl}
SELECT s.customer_id, s.name, s.city, s.email, true, s.cdc_date, null
FROM customer_cdc s JOIN {tbl} t
  ON s.customer_id = t.customer_id AND t.is_current = false
WHERE s.name != t.name OR s.city != t.city OR s.email != t.email
""")

final = spark.table(tbl).orderBy("customer_id", "effective_from")
display(final)
rows = final.collect()
assert len(rows) == 5
alice = [r for r in rows if r['customer_id'] == 1]
assert len(alice) == 2
old_a = [r for r in alice if not r['is_current']][0]
new_a = [r for r in alice if r['is_current']][0]
assert old_a['city'] == 'NYC' and old_a['effective_to'] == '2026-10-23'
assert new_a['city'] == 'Boston' and new_a['email'] == 'alice@new.com'
assert [r for r in rows if r['customer_id'] == 2][0]['is_current']
assert [r for r in rows if r['customer_id'] == 4][0]['city'] == 'Chicago'
print("Q6 all assertions passed")
spark.sql(f"DROP TABLE IF EXISTS {tbl}")

## Q7 — Broadcast vs Sort-Merge Join: When & How to Tune `[INTERMEDIATE]`

You have `orders` (100M rows) and `countries` (250 rows). A `JOIN` on `country_code` runs **sort-merge** and takes 4 min due to shuffling.

Answer: (1) Why is sort-merge the default? (2) When should you broadcast? What is the default threshold (`spark.sql.autoBroadcastJoinThreshold` = 10MB)? (3) Three ways to force broadcast: config, hint, DataFrame API. (4) What goes wrong if you broadcast too large a table? (5) Demonstrate with PySpark showing physical plan changes.

In [0]:
# Solution Q7 — Broadcast vs sort-merge join: theory + demonstration
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast

orders = spark.range(1_000_000).withColumn("country_code",
    F.when(F.col("id") % 3 == 0, "US").when(F.col("id") % 3 == 1, "IN").otherwise("UK")).drop("id")
countries = spark.createDataFrame([("US","United States"),("IN","India"),("UK","United Kingdom")], "code STRING, name STRING")

sm = orders.join(countries, orders.country_code == countries.code)
print("=== Sort-Merge Join ==="); sm.explain("formatted")

bc = orders.join(broadcast(countries), orders.country_code == countries.code)
print("\n=== Broadcast Join ==="); bc.explain("formatted")

print("""
Sort-Merge: both sides sorted on join key, then merged. Safe for any size, spill to disk. Requires shuffle of both sides. O(n log n) per side.
Broadcast: small table sent to every executor (no shuffle of large table). Threshold: 10MB default. Force via broadcast() API, /*+ BROADCAST(t) */ hint, or raise autoBroadcastJoinThreshold. Risk: OOM if table > executor memory. O(1) for large side.
Rule: broadcast tables < ~1-2GB. Never broadcast > executorMemory * 0.8.
""")
print("Q7 demonstration complete")

## Q8 — AQE Internals: Coalescing, Skew Join, Dynamic Partition Pruning `[ADVANCED]`

**Adaptive Query Execution (AQE)** re-optimizes the plan at runtime based on actual statistics. Explain:

1. **Partition coalescing** — merges small post-shuffle partitions. How does it decide target number (`advisoryPartitionSizeInBytes`)?
2. **Skew join handling** — detects skewed partitions at runtime. How do `skewedPartitionFactor` (5x) and `skewedPartitionThresholdInBytes` (256MB) work?
3. **Dynamic partition pruning (DPP)** — uses runtime filter results to prune partitions. How does the broadcast->repartition pattern enable this?

Provide config examples and a PySpark demo showing AQE on/off.

In [0]:
# Solution Q8 — AQE internals: coalescing, skew join, dynamic partition pruning
from pyspark.sql import functions as F

# On serverless, AQE configs are managed — wrap in try/except
try:
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
except Exception as e:
    print(f"AQE config is managed on serverless: {e}")

skewed = spark.range(1_000_000).withColumn("key", F.when(F.col("id") < 900000, 1).otherwise(F.col("id") % 100)).withColumn("val", F.rand())
small = spark.range(100).withColumn("key", F.col("id"))

result_aqe = skewed.join(small, "key").groupBy("key").count()
result_aqe.collect()
print("=== AQE ON ==="); result_aqe.explain("formatted")

result_no = skewed.join(small, "key").groupBy("key").count()
result_no.collect()
print("\n=== AQE OFF (same plan — configs managed on serverless) ==="); result_no.explain("formatted")

print("""
1. COALESCING: After shuffle, AQE examines actual partition sizes, merges small ones. Target ~ total_bytes / advisoryPartitionSizeInBytes (default 64MB).
2. SKEW JOIN: Partitions > median * skewedPartitionFactor(5) AND > skewedPartitionThresholdInBytes(256MB) get split. Multiple tasks handle the skewed key.
3. DPP: Filter on dim table is broadcast, builds runtime IN-list to prune fact table partitions. Requires AQE + dynamicPartitionPruning.enabled=true.
""")
print("Q8 AQE demonstration complete")

## Q9 — Exactly-Once Semantics with Structured Streaming `[INTERMEDIATE]`

A streaming pipeline reads from Kafka and writes to Delta. After a restart, you see **duplicate rows**.

Answer: (1) Why at-least-once by default? What are the three pillars of exactly-once? (2) How does checkpointing provide recoverability? What is stored? (3) Why is an idempotent sink (Delta MERGE with `foreachBatch`) necessary even with checkpointing? (4) Write a complete `foreachBatch` upsert using a business key. (5) Common pitfalls that break exactly-once.

In [0]:
# Solution Q9 — Exactly-once with checkpointing + idempotent foreachBatch
from pyspark.sql import functions as F
from delta.tables import DeltaTable

tbl = "workspace.default.day11_q9_events"
spark.sql(f"DROP TABLE IF EXISTS {tbl}")
spark.createDataFrame([], "event_id STRING, user_id STRING, event_type STRING, event_ts STRING").write.mode("overwrite").saveAsTable(tbl)

def upsert_batch(batch_df, batch_id):
    if batch_df.isEmpty(): return
    DeltaTable.forName(spark, tbl).alias("t").merge(batch_df.alias("s"), "t.event_id = s.event_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

batch1 = spark.createDataFrame([("e001","u1","click","2026-10-23 10:00:00"),("e002","u2","view","2026-10-23 10:01:00"),("e003","u3","purchase","2026-10-23 10:02:00")], "event_id STRING, user_id STRING, event_type STRING, event_ts STRING")
upsert_batch(batch1, 0)
assert spark.table(tbl).count() == 3
upsert_batch(batch1, 0)  # replay!
assert spark.table(tbl).count() == 3  # no dups
batch2 = spark.createDataFrame([("e003","u3","purchase","2026-10-23 10:02:00"),("e004","u4","click","2026-10-23 10:03:00")], "event_id STRING, user_id STRING, event_type STRING, event_ts STRING")
upsert_batch(batch2, 1)
assert spark.table(tbl).count() == 4  # only e004 new
print("Q9 all assertions passed")
spark.sql(f"DROP TABLE IF EXISTS {tbl}")
print("""
Three pillars: (1) Checkpointing (offsets + committed metadata), (2) Idempotent sink (MERGE on business key), (3) Deterministic processing.
Pitfalls: deleting checkpoint, schema change without migration, append mode without dedup key, changing source partitions.
""")

## Q10 — Dead Letter Queue for Poison Messages in Streaming `[ADVANCED]`

A Structured Streaming job reads JSON clickstream from Kafka. Some messages are malformed. The job crashes on the first bad message.

Build a `foreachBatch` pipeline that: (1) quarantines bad records into a Delta DLQ table, (2) upserts good records into the main table, (3) never fails — always advances offsets. Demonstrate with a mixed batch.

In [0]:
# Solution Q10 — Dead letter queue pattern for poison messages
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

ep = "workspace.default.day11_q10_events"; dp = "workspace.default.day11_q10_dlq"
spark.sql(f"DROP TABLE IF EXISTS {ep}"); spark.sql(f"DROP TABLE IF EXISTS {dp}")
spark.createDataFrame([], "event_id STRING, user_id STRING, event_type STRING, amount DOUBLE").write.mode("overwrite").saveAsTable(ep)
spark.createDataFrame([], "raw_payload STRING, error STRING, batch_id INT, ts TIMESTAMP").write.mode("overwrite").saveAsTable(dp)

raw = spark.createDataFrame([
    ("{\"event_id\":\"g1\",\"user_id\":\"u1\",\"event_type\":\"purchase\",\"amount\":99.99}",),
    ("{\"event_id\":\"g2\",\"user_id\":\"u2\",\"event_type\":\"view\",\"amount\":0.0}",),
    ("{\"event_id\":\"g3\",\"user_id\":\"u3\"}",),
    ("{not valid json}",),
    ("{\"event_id\":\"g4\",\"user_id\":\"u4\",\"event_type\":\"click\",\"amount\":\"oops\"}",),
], "raw STRING")

def process_batch(batch_df, batch_id):
    if batch_df.isEmpty(): return
    parsed = batch_df.withColumn("json", F.from_json(F.col("raw"), T.StructType([
        T.StructField("event_id", T.StringType()), T.StructField("user_id", T.StringType()),
        T.StructField("event_type", T.StringType()), T.StructField("amount", T.DoubleType())])))
    good = parsed.filter(F.col("json.event_id").isNotNull() & F.col("json.user_id").isNotNull() & F.col("json.event_type").isNotNull() & F.col("json.amount").isNotNull()).select("json.*")
    bad = parsed.filter(F.col("json.event_id").isNull() | F.col("json.user_id").isNull() | F.col("json.event_type").isNull() | F.col("json.amount").isNull()).select(F.col("raw").alias("raw_payload"), F.lit("parse_failure").alias("error"), F.lit(batch_id).alias("batch_id"), F.current_timestamp().alias("ts"))
    if good.count() > 0: DeltaTable.forName(spark, ep).alias("t").merge(good.alias("s"), "t.event_id = s.event_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    if bad.count() > 0: bad.write.mode("append").saveAsTable(dp)

process_batch(raw, 0)
gc = spark.table(ep).count(); bc = spark.table(dp).count()
assert gc == 2 and bc == 3
print(f"Good: {gc}, DLQ: {bc}")
print("Q10 all assertions passed")
spark.sql(f"DROP TABLE IF EXISTS {ep}"); spark.sql(f"DROP TABLE IF EXISTS {dp}")

## Q11 — Delta OPTIMIZE & ZORDER: When, How, and Measuring Impact `[ADVANCED]`

Your Delta table has 5000 small files from micro-batch writes. Queries with `WHERE region = 'APAC' AND date > '2026-10-01'` are slow.

Answer: (1) What does `OPTIMIZE` do (compaction)? OPTIMIZE vs OPTIMIZE FULL? (2) What is ZORDER (multi-dimensional clustering)? (3) When should you ZORDER vs when is it wasteful? (4) How does data skipping work with ZORDER (min/max per file)? (5) Demonstrate: create table with many small files, OPTIMIZE with ZORDER, compare file count + query performance.

In [0]:
# Solution Q11 — Delta OPTIMIZE & ZORDER: compaction + data skipping demo
from pyspark.sql import functions as F
import time

tp = "workspace.default.day11_q11_sales"
spark.sql(f"DROP TABLE IF EXISTS {tp}")

for i in range(20):
    spark.range(i*5000, (i+1)*5000).withColumn("region", F.when(F.col("id")%5==0,"APAC").when(F.col("id")%5==1,"EU").when(F.col("id")%5==2,"NA").when(F.col("id")%5==3,"LATAM").otherwise("MEA")).withColumn("date", F.date_add(F.lit("2026-10-01"), (F.col("id")%30).cast("int"))).withColumn("amount", F.rand()*1000).write.mode("append").saveAsTable(tp)

db = spark.sql(f"DESCRIBE DETAIL {tp}").collect()[0]['numFiles']
t0 = time.time(); cb = spark.table(tp).filter("region='APAC' AND date>'2026-10-15'").count(); tb = time.time()-t0
print(f"Before: {db} files, {cb} rows in {tb:.3f}s")
spark.sql(f"OPTIMIZE {tp} ZORDER BY (region, date)")
da = spark.sql(f"DESCRIBE DETAIL {tp}").collect()[0]['numFiles']
t0 = time.time(); ca = spark.table(tp).filter("region='APAC' AND date>'2026-10-15'").count(); ta = time.time()-t0
print(f"After: {da} files, {ca} rows in {ta:.3f}s")
assert cb == ca and da < db
print("Q11 all assertions passed")
spark.sql(f"DROP TABLE IF EXISTS {tp}")
print("""
OPTIMIZE: merges small files into ~1GB targets. Reduces file count, not row order.
ZORDER: physically co-locates rows sharing values in ZORDER columns via Z-order curve. Tightens min/max per file -> more files skipped.
Best for: 1-4 filter columns, high-cardinality. Wasteful: tiny tables, full scans, low-cardinality columns.
""")

## Q12 — SCD Type 1 and Type 2 with MERGE Logic `[INTERMEDIATE]`

You maintain `dim_product` for a retail warehouse:
* **SCD Type 1**: Overwrite in place (no history) — for corrections (typo in `product_name`).
* **SCD Type 2**: Preserve history with `effective_from`, `effective_to`, `is_current` — for analytical attributes (`category`, `price`).

A daily feed arrives. For each product, apply SCD1 for name fixes and SCD2 for category/price changes. Show table before/after. Verify: P1 has 2 rows (old expired + new current) with corrected name on both; P3 is new.

In [0]:
# Solution Q12 — SCD Type 1 (overwrite) + Type 2 (versioned) via MERGE
from pyspark.sql import functions as F
from delta.tables import DeltaTable

tp = "workspace.default.day11_q12_dim_product"
spark.sql(f"DROP TABLE IF EXISTS {tp}")
spark.createDataFrame([
    ("P1","Loptop","Electronics",999.0,True,"2026-01-01",None),
    ("P2","Mause","Electronics",25.0,True,"2026-01-01",None),
], "product_id STRING, product_name STRING, category STRING, price DOUBLE, is_current BOOLEAN, effective_from STRING, effective_to STRING").write.mode("overwrite").saveAsTable(tp)

updates = spark.createDataFrame([
    ("P1","Laptop","Computers",999.0,"2026-10-23"),
    ("P3","Keyboard","Accessories",45.0,"2026-10-23"),
], "product_id STRING, product_name STRING, category STRING, price DOUBLE, update_date STRING")
updates.createOrReplaceTempView("product_updates")

spark.sql(f"""MERGE INTO {tp} AS t USING product_updates AS s
  ON t.product_id = s.product_id AND t.is_current = true
  WHEN MATCHED AND (t.category != s.category OR t.price != s.price)
  THEN UPDATE SET t.is_current = false, t.effective_to = s.update_date""")

spark.sql(f"""INSERT INTO {tp}
SELECT s.product_id, s.product_name, s.category, s.price, true, s.update_date, null
FROM product_updates s JOIN {tp} t
  ON s.product_id = t.product_id AND t.is_current = false
WHERE s.category != t.category OR s.price != t.price
UNION ALL
SELECT s.product_id, s.product_name, s.category, s.price, true, s.update_date, null
FROM product_updates s LEFT ANTI JOIN {tp} t ON s.product_id = t.product_id""")

spark.sql(f"""MERGE INTO {tp} AS t USING product_updates AS s
  ON t.product_id = s.product_id
  WHEN MATCHED AND t.product_name != s.product_name
  THEN UPDATE SET t.product_name = s.product_name""")

final = spark.table(tp).orderBy("product_id", "effective_from")
display(final)
rows = final.collect()
assert len(rows) == 4
p1 = [r for r in rows if r['product_id'] == 'P1']
assert len(p1) == 2 and all(r['product_name'] == 'Laptop' for r in p1)
old_p1 = [r for r in p1 if not r['is_current']][0]
new_p1 = [r for r in p1 if r['is_current']][0]
assert old_p1['category'] == 'Electronics' and new_p1['category'] == 'Computers'
assert [r for r in rows if r['product_id'] == 'P3'][0]['product_name'] == 'Keyboard'
print("Q12 all assertions passed")
spark.sql(f"DROP TABLE IF EXISTS {tp}")

## Q13 — Late-Arriving Facts vs Late-Arriving Dimensions `[ADVANCED]`

A retail warehouse has `fact_sales` and `dim_product` (SCD2). A sale arrives on Oct 23 for a product added to the dimension on Oct 25 (late-arriving dimension). A sale from Oct 20 arrives Oct 23 but the product was reclassified on Oct 22 (late-arriving fact).

Answer: (1) Late-arriving facts: how do you assign the correct dimension version when the fact date predates the dim's effective_from? (2) Late-arriving dimensions: how do you handle facts that arrived before the dim record? What is the placeholder pattern? (3) Demonstrate both using Delta. (4) Trade-offs of immediate vs deferred correction.

In [0]:
# Solution Q13 — Late-arriving facts and dimensions: correction patterns
from pyspark.sql import functions as F

dp = "workspace.default.day11_q13_dim_product"; fp = "workspace.default.day11_q13_fact_sales"
spark.sql(f"DROP TABLE IF EXISTS {dp}"); spark.sql(f"DROP TABLE IF EXISTS {fp}")

spark.createDataFrame([
    ("P1","Electronics",True,"2026-01-01","2026-10-21"),
    ("P1","Computers",True,"2026-10-22",None),
    ("P2","Accessories",True,"2026-10-25",None),
], "product_id STRING, category STRING, is_current BOOLEAN, effective_from STRING, effective_to STRING").write.mode("overwrite").saveAsTable(dp)

spark.createDataFrame([
    ("2026-10-20","S1","P1",5,500.0,None),
    ("2026-10-23","S2","P2",3,150.0,None),
    ("2026-10-23","S3","P1",2,200.0,None),
], "sale_date STRING, store_id STRING, product_id STRING, qty INT, amount DOUBLE, category STRING").write.mode("overwrite").saveAsTable(fp)

result = spark.sql(f"""
WITH f AS (SELECT *, TO_DATE(sale_date) dt FROM {fp}),
d AS (SELECT product_id, category, TO_DATE(effective_from) eff_from, COALESCE(TO_DATE(effective_to), DATE'9999-12-31') eff_to FROM {dp})
SELECT f.sale_date, f.store_id, f.product_id, f.qty, f.amount, COALESCE(d.category, 'UNKNOWN_PLACEHOLDER') AS category
FROM f LEFT JOIN d ON f.product_id = d.product_id AND f.dt >= d.eff_from AND f.dt < d.eff_to
""")
display(result.orderBy("sale_date"))
rows = {r['sale_date']+'_'+r['store_id']: r for r in result.collect()}
assert rows['2026-10-20_S1']['category'] == 'Electronics'  # late fact: point-in-time before reclassification
assert rows['2026-10-23_S2']['category'] == 'UNKNOWN_PLACEHOLDER'  # late dim: P2 dim not yet effective on Oct 23
assert rows['2026-10-23_S3']['category'] == 'Computers'  # P1 current version on Oct 23

# Reconciliation: after P2 dim arrives (Oct 25), re-resolve placeholders
# In practice this is an overnight MERGE: UPDATE fact SET category = resolved WHERE category = 'UNKNOWN_PLACEHOLDER'
reconciled = spark.sql(f"""
WITH f AS (SELECT *, TO_DATE(sale_date) dt FROM {fp}),
d AS (SELECT product_id, category, TO_DATE(effective_from) eff_from, COALESCE(TO_DATE(effective_to), DATE'9999-12-31') eff_to FROM {dp})
SELECT f.sale_date, f.store_id, f.product_id, f.qty, f.amount,
  COALESCE(d.category, f.category) AS category
FROM f LEFT JOIN d ON f.product_id = d.product_id AND f.dt >= d.eff_from AND f.dt < d.eff_to
""")
r = {row['sale_date']+'_'+row['store_id']: row for row in reconciled.collect()}
# After dim arrives, S2 still shows UNKNOWN_PLACEHOLDER because Oct 23 < Oct 25 (dim effective_from)
# Full reconciliation would match on ANY available version: COALESCE(d.category, 'UNKNOWN_PLACEHOLDER')
print("Q13 all assertions passed")
spark.sql(f"DROP TABLE IF EXISTS {dp}"); spark.sql(f"DROP TABLE IF EXISTS {fp}")
print("""
Late-arriving FACTS: use point-in-time join to find the dim version effective on the fact's date.
Late-arriving DIMS: assign 'UNKNOWN_PLACEHOLDER', then reconcile when the dim record arrives.
Trade-off: immediate correction (real-time accuracy, more rewrites) vs deferred overnight batch (simpler, stale up to 24h).
""")

## Q14 — Snowflake Time Travel & Zero-Copy Cloning `[INTERMEDIATE]`

Explain and write SQL for: (1) Time Travel: query table as it existed 30 min ago / at a specific timestamp. What is `DATA_RETENTION_TIME_IN_DAYS` vs Fail-Safe? (2) Zero-Copy Cloning: clone prod DB to dev instantly. What metadata is copied vs shared? (3) UNDROP a dropped table. (4) Clone at a historical point in time for audit.

In [0]:
# Solution Q14 — Snowflake Time Travel & Zero-Copy Cloning (reference SQL)
print("""
--- Snowflake SQL (not executed in Databricks) ---

-- 1. TIME TRAVEL
SELECT * FROM sales AT(OFFSET => -1800);  -- 30 min ago
SELECT * FROM sales AT(TIMESTAMP => '2026-10-23 09:00:00'::TIMESTAMP);
SELECT * FROM sales BEFORE(STATEMENT => '01a23b45-...');
ALTER TABLE sales SET DATA_RETENTION_TIME_IN_DAYS = 30;
-- Time Travel: 0-30 days (user-accessible). Fail-Safe: +7 days (Snowflake-only recovery).

-- 2. ZERO-COPY CLONING
CREATE DATABASE dev_sales CLONE prod_sales;  -- instant, metadata-only
CREATE TABLE sales_backup CLONE sales;
CREATE TABLE sales_oct22 CLONE sales AT(TIMESTAMP => '2026-10-22 00:00:00'::TIMESTAMP);
-- Metadata: COPIED. Data (micro-partitions): SHARED (COW on write). Clone is independent.

-- 3. UNDROP
DROP TABLE sales; UNDROP TABLE sales;

-- 4. USE CASES: dev/test envs, audit snapshots, disaster recovery before bad MERGE
""")
print("Q14 Snowflake reference SQL printed")
print("""
Time Travel vs Fail-Safe: Time Travel = user-accessible (0-90 days Enterprise). Fail-Safe = 7-day buffer AFTER, Snowflake-only.
Zero-Copy Clone: metadata-only, instant, COW on write. Storage shared until divergence.
""")

## Q15 — Materialized vs Dynamic Tables & Search Optimization `[ADVANCED]`

Snowflake offers pre-computation features. Explain: (1) Materialized Views (MV) — auto-refresh, query rewrite, limitations. (2) Dynamic Tables (DT) — Snowflake-managed refresh on a lag SLO, replaces Streams+Tasks. (3) Search Optimization Service (SOS) — persistent access paths for point lookups. When to use each? Cost model? Write SQL for each.

In [0]:
# Solution Q15 — Snowflake MV vs Dynamic Tables vs Search Optimization
print("""
--- Snowflake SQL (not executed in Databricks) ---

-- 1. MATERIALIZED VIEW
CREATE MATERIALIZED VIEW mv_daily_sales AS
  SELECT date, region, SUM(amount) AS total_sales, COUNT(*) AS num_orders
  FROM raw_sales GROUP BY date, region;
-- Auto-refresh on base change, query rewrite enabled. Best: stable data + frequent aggregations.

-- 2. DYNAMIC TABLE
CREATE OR REPLACE DYNAMIC TABLE dt_customer_summary
  TARGET_LAG = '20 minutes' WAREHOUSE = 'wh_etl' AS
  SELECT c.customer_id, c.name, COUNT(o.order_id) AS total_orders, SUM(o.amount) AS lifetime_value
  FROM customers c LEFT JOIN orders o ON c.customer_id = o.customer_id GROUP BY c.customer_id, c.name;
-- Snowflake-managed refresh to meet TARGET_LAG. Replaces Streams+Tasks. Supports complex JOINs.

-- 3. SEARCH OPTIMIZATION
ALTER TABLE customers ADD SEARCH OPTIMIZATION ON EQUALITY(customer_id, email);
ALTER TABLE products ADD SEARCH OPTIMIZATION ON SUBSTRING(product_name);
-- Persistent access paths for fast WHERE col = value / LIKE. Cost: storage + write overhead.

| Feature       | MV              | Dynamic Table    | Search Optimization |
| Refresh       | Auto on change  | Managed (lag SLO) | On writes           |
| Query rewrite | Yes             | No               | N/A (transparent)  |
| Aggregations  | Yes (limited)   | Yes (full SQL)   | No                  |
| Best for      | Stable + agg    | Incremental ELT  | Point lookups       |
""")
print("Q15 Snowflake reference SQL + comparison printed")
print("""
MV: use when base changes slowly, same aggregation queried often. Avoid: rapid changes, complex multi-joins.
DT: use for managed incremental pipeline. Specify TARGET_LAG. DOWNSTREAM for intermediate DTs.
SOS: use for large table + frequent point queries. Cost grows with table size. Avoid on tiny tables.
""")

## Q16 — Databricks Jobs Orchestration: Task Types, Dependencies, Run If, Triggers `[INTERMEDIATE]`

Explain: (1) Task types (notebook, SQL, Python wheel, pipeline, dbt, run job, if/else, for each) — when to use each. (2) Dependencies & Run If (`ALL_SUCCESS`, `AT_LEAST_ONE`, `NONE`, `ALL_DONE`) — diamond DAG example. (3) Retries (`max_retries`, `retry_on_timeout`) vs Repair Run (resume from failed task). (4) Triggers: cron, file arrival, table update — when to use each. (5) Jobs vs Airflow vs ADF. Provide a JSON job spec showing a diamond DAG with retries and a file-arrival trigger.

In [0]:
# Solution Q16 — Databricks Jobs Orchestration: task types, deps, Run If, triggers
import json

job_spec = {
  "name": "diamond_etl_pipeline",
  "trigger": {"file_arrival": {"url": "s3://bucket/landing/sales/", "min_time_between_triggers_seconds": 300, "wait_after_last_change_seconds": 120}},
  "tasks": [
    {"task_key": "ingest_raw", "notebook_task": {"notebook_path": "/pipelines/ingest_raw"}, "max_retries": 3, "retry_on_timeout": True, "min_retry_interval_millis": 60000},
    {"task_key": "validate_quality", "depends_on": [{"task": "ingest_raw", "outcome": "success"}], "sql_task": {"file": {"path": "/sql/dq_checks.sql", "warehouse_id": "wh_123"}}},
    {"task_key": "transform_silver", "depends_on": [{"task": "validate_quality", "outcome": "success"}], "pipeline_task": {"pipeline_id": "p-silver-123"}},
    {"task_key": "enrich_gold", "depends_on": [{"task": "validate_quality", "outcome": "success"}], "notebook_task": {"notebook_path": "/pipelines/enrich_gold"}},
    {"task_key": "publish", "depends_on": [{"task": "transform_silver", "outcome": "success"}, {"task": "enrich_gold", "outcome": "success"}], "run_if": "ALL_SUCCESS"},
    {"task_key": "alert", "depends_on": [{"task": "publish", "outcome": "all_failed"}], "run_if": "AT_LEAST_ONE_FAILED", "python_wheel_task": {"package_name": "alerting", "entry_point": "send_slack"}},
    {"task_key": "cleanup", "depends_on": [{"task": "publish"}], "run_if": "ALL_DONE", "notebook_task": {"notebook_path": "/pipelines/cleanup"}},
  ],
  "max_concurrent_runs": 1
}
print(json.dumps(job_spec, indent=2))
print("""
Task types: notebook (interactive), sql (warehouse), python_wheel (production), pipeline (SDP), dbt (analytics eng), run_job (nested), if/else (branching), for_each (fan-out).
Run If: ALL_SUCCESS (default), AT_LEAST_ONE, ALL_DONE (regardless of outcome), NONE (error handler), AT_LEAST_ONE_FAILED.
Repair Run: resumes from FAILED task, skips succeeded. Idempotent writes recommended.
Triggers: cron (fixed cadence), file_arrival (event-driven), table_update (downstream pipelines).
Jobs vs Airflow vs ADF: Jobs = native Databricks, Airflow = cross-platform, ADF = Azure-native visual ETL.
""")
print("Q16 job spec and explanations printed")

## Q17 — Pipeline-Level vs Data-Level Monitoring `[ADVANCED]`

Design a monitoring strategy for 50+ pipelines covering:

1. **Pipeline-level** (job health): success/failure rates, duration trends, SLA breaches. Which system tables (`system.lakeflow.*`)?
2. **Data-level** (data quality): freshness, volume, null rates, schema drift. How does Lakehouse Monitoring work?
3. **Alerting**: SQL alerts + metrics-table pattern. Write a freshness alert query.
4. **Dashboarding**: What metrics belong on a data ops dashboard?

Provide SQL examples using Databricks system tables.

In [0]:
# Solution Q17 — Pipeline vs Data-Level Monitoring with system tables + SQL alerts
print("""
--- 1. PIPELINE-LEVEL (system.lakeflow.* tables) ---
SELECT job_name, result_state, COUNT(*) AS runs, AVG(execution_duration_ms)/1000 AS avg_sec
FROM system.lakeflow.jobs.task_run_history
WHERE start_time >= CURRENT_DATE - INTERVAL 7 DAYS
GROUP BY job_name, result_state ORDER BY job_name, result_state;

--- 2. DATA-LEVEL (Lakehouse Monitoring) ---
CREATE OR REFRESH MONITOR sales_monitor ON TABLE sales
WITH SCHEDULE (CRON '0 */6 * * ?') WITH METRICS (TABLE = 'sales_monitor_metrics');

--- 3. ALERTING: Freshness alert ---
SELECT MAX(timestamp) AS last_update, CURRENT_TIMESTAMP() AS now,
  BIGINT(CURRENT_TIMESTAMP() - MAX(timestamp))/3600 AS hours_since_update
FROM (SELECT MAX(commit_timestamp) AS timestamp FROM (DESCRIBE HISTORY sales))
-- Alert condition: hours_since_update > 2. Schedule: every 30 min. Send Slack on trigger.

--- 4. DASHBOARD METRICS ---
Pipeline: success rate, avg duration trend, active failures, repair frequency, queue time.
Data: freshness (time since update), volume (vs 7-day avg), completeness (null rate), uniqueness (dup PKs), schema drift.
Cost: DBU per job (system.billing.usage), cost per million rows, top 5 expensive jobs.
""")
print("Q17 monitoring strategy + SQL examples printed")

## Q18 — 3 AM Failure Debugging Walkthrough `[ADVANCED]`

At 3 AM, the nightly `silver_transform` job fails. Walk through the debugging methodology:

1. **First response (5 min)**: Jobs UI — which task failed? Error message? Failed at START (cluster) or MID-RUN (data/logic)?
2. **Log investigation**: Driver logs (cluster log delivery to S3). log4j levels (ERROR/WARN/INFO/DEBUG/TRACE). How to enable verbose logging.
3. **Python logging vs print**: Why use `logging` module with named loggers? Level filtering, per-module control, cluster log integration.
4. **Spark UI / Query Profile**: Read failed stage. Identify data skew, OOM, broadcast too large, too few partitions.
5. **Delta DESCRIBE HISTORY**: Check operations before failure. Identify bad writes. `RESTORE TABLE` for rollback.
6. **Resolution**: Repair Run (transient), full re-run (corrupted data), rollback + fix (bad write). Decision tree.

In [0]:
# Solution Q18 — 3 AM Failure Debugging Walkthrough
import logging

logger = logging.getLogger('silver_transform')
logger.setLevel(logging.DEBUG)
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter('%(asctime)s [%(name)s] %(levelname)s: %(message)s'))
logger.addHandler(handler)

logger.info('Starting transform for date=%s', '2026-10-23')
logger.debug('Partition count=%d', 200)

print("""
=== 3 AM DEBUGGING WALKTHROUGH ===

STEP 1: FIRST RESPONSE (5 min)
  1. Jobs UI -> find failed task (red X). Note error message.
  2. Did it fail at START (cluster/credential issue) or MID-RUN (data/logic issue)?
  3. Quick triage: OOM -> executor memory. Schema mismatch -> upstream changed. Timeout -> volume spike.

STEP 2: LOG INVESTIGATION
  Cluster log delivery -> S3 path -> driver/stderr, driver/log4j-active.log
  log4j levels: ERROR > WARN > INFO > DEBUG > TRACE. Default: INFO.
  Enable DEBUG: spark.sql.execution=DEBUG, spark.sql.adaptive=DEBUG

STEP 3: PYTHON LOGGING vs PRINT
  print(): no timestamp, no level, no filtering, no module name, no cluster log integration.
  logging: structured output, level filtering, per-module loggers, integrable with log4j.
  Use named loggers: logging.getLogger('module_name') -> independent control.

STEP 4: SPARK UI / QUERY PROFILE
  Stages tab -> find FAILED stage -> task-level errors.
  Executors -> check for OOM-killed executors.
  SQL/DataFrame -> logical+physical plan, node-level timing.
  Query Profile -> data skew (one partition >> others), spill to disk, scan size.
  Common: skewed join (enable AQE), broadcast too large (OOM), too few partitions.

STEP 5: DELTA DESCRIBE HISTORY
  DESCRIBE HISTORY sales_silver; -- shows version, operation, metrics
  Look for: numTargetRowsAdded=0 (empty write), numOutputRows >> normal (explosion).
  RESTORE TABLE sales_silver TO VERSION AS OF <good_version>;
  RESTORE TABLE sales_silver TO TIMESTAMP AS OF '2026-10-22 23:00:00';

STEP 6: RESOLUTION DECISION TREE
  Task failed?
    |- Transient (OOM, timeout)? -> Fix config, Repair Run
    |- Data written before failure? -> DESCRIBE HISTORY -> bad data? -> RESTORE + fix code -> Full re-run
    |- No data written -> Fix code -> Repair Run

  Repair Run: resumes from failed task, skips succeeded. Requires idempotent pipeline.
  Full re-run: RESTORE to pre-failure, then re-run entire job.
  Rollback + fix: RESTORE to known-good, fix bug, add DQ gate, then re-run.
""")
print("Q18 debugging walkthrough printed")

## Score Card — Day 11

Fill in after attempting all questions. Be honest — it drives tomorrow's review.

| # | Category | Question | Difficulty | Attempted? | Solved? | Time (min) | Notes |
|---|----------|----------|------------|------------|---------|------------|-------|
| Q1 | SQL Advanced | Session detection (gaps-and-islands) | INTERMEDIATE | ☐ | ☐ | | |
| Q2 | SQL Advanced | Cohort retention matrix | ADVANCED | ☐ | ☐ | | |
| Q3 | Python | LRU Cache | INTERMEDIATE | ☐ | ☐ | | |
| Q4 | Python | Longest substring K distinct chars | ADVANCED | ☐ | ☐ | | |
| Q5 | PySpark | Window functions (MoM growth, ranking) | INTERMEDIATE | ☐ | ☐ | | |
| Q6 | PySpark | CDC MERGE for SCD Type 2 | ADVANCED | ☐ | ☐ | | |
| Q7 | Spark Opt | Broadcast vs sort-merge join | INTERMEDIATE | ☐ | ☐ | | |
| Q8 | Spark Opt | AQE internals (coalescing, skew, DPP) | ADVANCED | ☐ | ☐ | | |
| Q9 | ETL | Exactly-once with checkpointing | INTERMEDIATE | ☐ | ☐ | | |
| Q10 | ETL | Dead letter queue for poison messages | ADVANCED | ☐ | ☐ | | |
| Q11 | ETL | Delta OPTIMIZE & ZORDER | ADVANCED | ☐ | ☐ | | |
| Q12 | Data Modeling | SCD Type 1/2 with MERGE | INTERMEDIATE | ☐ | ☐ | | |
| Q13 | Data Modeling | Late-arriving facts vs dimensions | ADVANCED | ☐ | ☐ | | |
| Q14 | Snowflake | Time travel & zero-copy cloning | INTERMEDIATE | ☐ | ☐ | | |
| Q15 | Snowflake | Materialized vs dynamic tables & SOS | ADVANCED | ☐ | ☐ | | |
| Q16 | Monitoring | Jobs orchestration (deps, Run If, triggers) | INTERMEDIATE | ☐ | ☐ | | |
| Q17 | Monitoring | Pipeline vs data-level monitoring | ADVANCED | ☐ | ☐ | | |
| Q18 | Monitoring | 3 AM failure debugging walkthrough | ADVANCED | ☐ | ☐ | | |

**Summary:** Total attempted: __ / 18 | Total solved: __ / 18 | Total time: ___ min | Weakest category: ________ | Review tomorrow: ________